In [ ]:
# ============================================================
# CELL 1: Install Libraries & Imports
# ============================================================

# Install any missing packages (run once)
# !pip install scikit-learn tensorflow pandas numpy matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Conv1D, MaxPooling1D,
    Flatten, Dropout, RepeatVector, TimeDistributed
)
from tensorflow.keras.callbacks import EarlyStopping

# Fix random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ All libraries imported successfully!")
print(f"   TensorFlow version: {tf.__version__}")

In [ ]:
 # ============================================================
# CELL 2: Load & Explore the Dataset
# ============================================================
normal = pd.read_csv("../normal.csv")
attack = pd.read_csv("../attack.csv")
merged = pd.concat([normal, attack], ignore_index=True)

# ── Basic Exploration ──
print(f"\n📊 Dataset shape: {merged.shape}")
print(f"   Columns: {list(merged.columns[:5])} ... ({len(merged.columns)} total)")
print(f"\n📋 First 3 rows:")
merged.head(3)

In [ ]:
# ============================================================
# CELL 3: Preprocessing – Normalize + Train/Test Split
# ============================================================

# ── Step 3a: Identify label column ──

label_col = 'attack'

# ── Step 3b: Extract features and labels ──
# Drop non-numeric or timestamp columns if any
feature_cols = [c for c in merged.columns if c != label_col]
df_features = merged[feature_cols].select_dtypes(include=[np.number])

X = df_features.values
y_raw = merged[label_col].values

# ── Step 3c: Encode labels as binary (0=Normal, 1=Attack) ──
# Handle both string ('Normal','Attack') and numeric (0,1) labels
unique_labels = np.unique(y_raw)
print(f"   Unique label values: {unique_labels}")

if y_raw.dtype == object or isinstance(y_raw[0], str):
    # String labels
    y = np.where(y_raw == 'Normal', 0, 1).astype(int)
else:
    # Numeric – map to 0/1
    y = (y_raw != 0).astype(int)   # 0 stays 0; anything else → 1

print(f"   Normal samples : {np.sum(y == 0):,}")
print(f"   Attack samples : {np.sum(y == 1):,}")

# ── Step 3d: Handle missing values ──
X = np.nan_to_num(X, nan=0.0)

# ── Step 3e: Normalize using MinMaxScaler ──
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# ── Step 3f: Train/Test Split (80/20) ──
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✅ Preprocessing complete!")
print(f"   Features used  : {X_scaled.shape[1]}")
print(f"   Training size  : {X_train.shape[0]:,}")
print(f"   Testing  size  : {X_test.shape[0]:,}")

# ── Step 3g: Prepare 3D versions for LSTM/GRU/CNN ──
# Shape: (samples, timesteps=1, features)
X_train_3d = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_3d  = X_test.reshape(X_test.shape[0],  1, X_test.shape[1])

print(f"   3D shape (LSTM/GRU/CNN): {X_train_3d.shape}")

# ── Plot class distribution ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = [np.sum(y == 0), np.sum(y == 1)]
axes[0].bar(['Normal (0)', 'Attack (1)'], counts, color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Sample Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Feature heatmap (first 10 features)
sample_data = pd.DataFrame(X_scaled[:500, :10],
                            columns=[f'F{i}' for i in range(10)])
sns.heatmap(sample_data.T, ax=axes[1], cmap='YlOrRd', cbar=True)
axes[1].set_title('Feature Heatmap (first 500 samples, 10 features)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sample Index')

plt.tight_layout()
plt.savefig('preprocessing_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("📊 Chart saved as 'preprocessing_overview.png'")

In [ ]:
# ============================================================
# CELL 4: MODEL 1 – Autoencoder (Main Model)
# ============================================================
# How it works:
#   - Train ONLY on normal data (unsupervised)
#   - At test time, compute reconstruction error
#   - High error → anomaly (attack)

print("="*55)
print("  MODEL 1: AUTOENCODER")
print("="*55)

n_features = X_train.shape[1]

# ── Build Autoencoder ──
input_layer = Input(shape=(n_features,))

# Encoder
encoded = Dense(32, activation='relu')(input_layer)
encoded = Dense(16, activation='relu')(encoded)
encoded = Dense(8,  activation='relu')(encoded)   # Bottleneck

# Decoder
decoded = Dense(16, activation='relu')(encoded)
decoded = Dense(32, activation='relu')(decoded)
output_layer = Dense(n_features, activation='sigmoid')(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

# ── Train on NORMAL data only ──
X_train_normal = X_train[y_train == 0]
print(f"\n🔧 Training on {len(X_train_normal):,} normal samples...")

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_ae = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=20,
    batch_size=1024,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# ── Compute reconstruction error on test set ──
X_test_reconstructed = autoencoder.predict(X_test, verbose=0)
recon_errors = np.mean(np.power(X_test - X_test_reconstructed, 2), axis=1)

# ── Choose threshold (95th percentile of normal error) ──
X_train_normal_recon = autoencoder.predict(X_train_normal, verbose=0)
normal_errors = np.mean(np.power(X_train_normal - X_train_normal_recon, 2), axis=1)
threshold = np.percentile(normal_errors, 95)
print(f"\n   Anomaly threshold (95th percentile): {threshold:.6f}")

# ── Predict: error > threshold → Attack ──
y_pred_ae = (recon_errors > threshold).astype(int)

# ── Evaluate ──
acc_ae = accuracy_score(y_test, y_pred_ae)
f1_ae  = f1_score(y_test, y_pred_ae, zero_division=0)

print(f"\n📈 AUTOENCODER RESULTS")
print(f"   Accuracy : {acc_ae:.4f}  ({acc_ae*100:.2f}%)")
print(f"   F1 Score : {f1_ae:.4f}")
print("\n" + classification_report(y_test, y_pred_ae, target_names=['Normal', 'Attack']))

# ── Plot: Training Loss ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_ae.history['loss'],     label='Train Loss', color='steelblue', linewidth=2)
axes[0].plot(history_ae.history['val_loss'], label='Val Loss',   color='tomato',    linewidth=2, linestyle='--')
axes[0].set_title('Autoencoder – Training Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reconstruction error distribution
axes[1].hist(recon_errors[y_test == 0], bins=50, alpha=0.6, label='Normal', color='steelblue')
axes[1].hist(recon_errors[y_test == 1], bins=50, alpha=0.6, label='Attack', color='tomato')
axes[1].axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold={threshold:.4f}')
axes[1].set_title('Reconstruction Error Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Reconstruction Error (MSE)')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('autoencoder_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("📊 Chart saved as 'autoencoder_results.png'")

In [ ]:
print(confusion_matrix(y_test, y_pred_ae))

In [ ]:
# --- Test ONLY on attack.csv ---
print("Evaluating Autoencoder ONLY on attack.csv...")
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

atk_df = pd.read_csv("../attack.csv")
label_col = 'attack'
feature_cols = [c for c in atk_df.columns if c != label_col and c != 'Timestamp' and c != 'label']
X_atk = atk_df[feature_cols].select_dtypes(include=[np.number]).values
X_atk = np.nan_to_num(X_atk, nan=0.0)

# The scaler is already trained as 'scaler'
X_atk_scaled = scaler.transform(X_atk)

y_raw_atk = atk_df[label_col].values
if y_raw_atk.dtype == object or isinstance(y_raw_atk[0], str):
    y_atk = np.where(y_raw_atk == 'Normal', 0, 1).astype(int)
else:
    y_atk = (y_raw_atk != 0).astype(int)

# Predict
X_atk_reconstructed = autoencoder.predict(X_atk_scaled, verbose=0)
recon_errors_atk = np.mean(np.power(X_atk_scaled - X_atk_reconstructed, 2), axis=1)

# threshold is already defined in the notebook
y_pred_atk = (recon_errors_atk > threshold).astype(int)

print("Classification Report on attack.csv:")
print(classification_report(y_atk, y_pred_atk))
print("Confusion Matrix on attack.csv:")
print(confusion_matrix(y_atk, y_pred_atk))
